In [ ]:
# !pip install tensorflow

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf


In [ ]:
# df=pd.read_csv("/kaggle/input/datasets/mohakpandey/quotes/qoute_dataset.csv")

In [ ]:
# df.head()

In [ ]:
# data=df.drop(columns="Author")

In [3]:
with open("/kaggle/input/datasets/mohakpandey/wikitext/wiki.train.tokens", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(len(lines))
print(lines[:10])

1801350
[' \n', ' = Valkyria Chronicles III = \n', ' \n', ' Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " <unk> Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more

In [4]:
clean_lines = []

for line in lines:
    line = line.strip()

    # Remove empty lines
    if line == "":
        continue

    # Remove headings
    if line.startswith("="):
        continue

    # Replace special tokens
    line = line.replace("<unk>", "")
    line = line.replace("@-@", "-")
    line = line.replace("@.@", ".")
    line = line.replace("@,@", ",")

    # Remove very short lines
    if len(line.split()) < 4:
        continue

    clean_lines.append(line.lower())

In [5]:
len(clean_lines)

839395

In [6]:
print(clean_lines[1])

the game began development in 2010 , carrying over a large portion of the work done on valkyria chronicles ii . while it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . character designer  honjou and composer hitoshi sakimoto both returned from previous entries , along with valkyria chronicles ii director takeshi ozawa . a large team of writers handled the script . the game 's opening theme was sung by may 'n .


In [8]:
import random

random.seed(42)
random.shuffle(clean_lines)

clean_lines = clean_lines[:100000]

In [10]:
clean_lines = [
    line for line in clean_lines
    if 5 <= len(line.split()) <= 80
]

In [11]:
import numpy as np

lengths = [len(x.split()) for x in clean_lines]

print("Maximum:", max(lengths))
print("Average:", np.mean(lengths))
print("95%:", np.percentile(lengths,95))
print("99%:", np.percentile(lengths,99))

Maximum: 80
Average: 40.1980001829101
95%: 77.0
99%: 80.0


WikiText
      ↓
Clean
      ↓
Split into sentences ⭐
      ↓
Shuffle
      ↓
Take 100k sentences
      ↓
Tokenizer(num_words=20000)
      ↓
Create sequences
      ↓
Pad
      ↓
Train LSTM

In [43]:
# max_len = int(np.percentile(lengths,99))
max_len=50

In [ ]:
# from IPython.utils import text
# df1 = pd.read_csv("/kaggle/input/datasets/mohakpandey/eng-sentences-tsv/eng_sentences.tsv", sep="\t",header=None,names=["id","lang","text"])

In [ ]:
# df1.head()

In [ ]:
# df1.shape

In [ ]:
# df1.columns

In [ ]:
# data=df1.drop(columns=["id",'lang'])

In [ ]:
# data.head()

In [ ]:
# data.shape

In [15]:
import string

In [ ]:
# text = data["text"][:50000]
# text=text.str.lower()

In [ ]:
# text = data["quote"]
# text=text.str.lower()

In [13]:
text=clean_lines

In [18]:
# translator =str.maketrans('','',string.punctuation)
# text=text.apply(lambda x:x.translate(translator))

In [33]:
# text.head()

In [34]:
import re

def clean_text(text):
    text = text.lower()

    # Replace numbers with a special token
    text = re.sub(r"\d+", "<NUM>", text)

    return text

In [35]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [36]:

vocab_size=20000
tokenize = Tokenizer(
    num_words=20000,
    oov_token="<OOV>"
)
# tokenize=Tokenizer(num_words=vocab_size)
tokenize.fit_on_texts(text)

In [37]:
word_index=tokenize.word_index
print(len(word_index))

63940


In [38]:
print("Original vocab:", len(tokenize.word_index))
print("Using vocab:", min(20000, len(tokenize.word_index)))

Original vocab: 63940
Using vocab: 20000


In [32]:
from collections import Counter

counts = tokenize.word_counts

freq = Counter(counts.values())

print("Words occurring once:", sum(v == 1 for v in counts.values()))
print("Words occurring twice:", sum(v == 2 for v in counts.values()))

Words occurring once: 27915
Words occurring twice: 10290


In [25]:
list(word_index.items())[:10]

[('the', 1),
 ('of', 2),
 ('and', 3),
 ('in', 4),
 ('a', 5),
 ('to', 6),
 ('was', 7),
 ('on', 8),
 ('as', 9),
 ('is', 10)]

In [41]:

sequence=tokenize.texts_to_sequences(text)

In [42]:
sequences = tokenize.texts_to_sequences(text)

max_index = max(max(seq) for seq in sequences if seq)

print(max_index)

19999


In [44]:
X=[]
y=[]

for seq in sequence:
  for i in range(1,len(seq)):
    X.append(seq[:i])
    y.append(seq[i])

In [45]:
len(X)

1098392

In [46]:
len(y)

1098392

In [ ]:
# max_len=max(len(x) for x in X)
# print(max_len)

In [47]:
sentence_lengths = [len(seq) for seq in X]

print("Maximum length:", max(sentence_lengths))
print("Average length:", np.mean(sentence_lengths))
print("95th percentile:", np.percentile(sentence_lengths, 95))
print("99th percentile:", np.percentile(sentence_lengths, 99))

Maximum length: 76
Average length: 24.489182368407636
95th percentile: 56.0
99th percentile: 64.0


In [48]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_pad=pad_sequences(X,maxlen=max_len,padding="pre")

In [49]:
X_pad

array([[    0,     0,     0, ...,     0,     0,   251],
       [    0,     0,     0, ...,     0,   251,  3648],
       [    0,     0,     0, ...,   251,  3648,  1762],
       ...,
       [    0,     0,     0, ...,   556,    13, 13605],
       [    0,     0,     0, ...,    13, 13605, 12290],
       [    0,     0,     0, ..., 13605, 12290,     4]], dtype=int32)

In [50]:
y=np.array(y)

In [51]:
X_pad.shape

(1098392, 50)

In [52]:
y.shape

(1098392,)

In [ ]:
# from tensorflow.keras.utils import to_categorical
# y_one_hot=to_categorical(y,num_classes=vocab_size)

In [53]:
y.shape

(1098392,)

In [ ]:
# y_one_hot.shape

In [55]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense

In [56]:
strategy = tf.distribute.MirroredStrategy()

with strategy.scope():

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.001,
        clipnorm=1.0
    )

    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=128,input_length=max_len
        ),

        tf.keras.layers.LSTM(
            256,dropout=0.2
            
        ),

        tf.keras.layers.Dense(
            vocab_size,
            activation="softmax"
        )
    ])

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1785838877.502366   23261 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785838877.507582   23261 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [57]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [58]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    verbose=1
)

# checkpoint = tf.keras.callbacks.ModelCheckpoint(
#     "best_model.keras",
#     monitor="val_loss",
#     save_best_only=True,
#     mode="min",
#     verbose=1
# )

history = model.fit(
    X_pad,
    y,
    epochs=100,
    batch_size=512,
    validation_split=0.2,
    callbacks=[reduce_lr,early_stop],verbose=1
    
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

In [59]:
model.save("wikitext_ac18_model.h5")

In [133]:
import tensorflow as tf

tf.keras.backend.clear_session()

In [134]:
from tensorflow.keras.models import load_model

model = load_model("/kaggle/working/wikitext_ac18_model.h5")

In [135]:

index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [136]:
print(tokenize.texts_to_sequences(["what are you"]))

[[327, 25, 185]]


In [137]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predictor(model, tokenizer, text, max_len, temperature=0.8, top_k=5):

    text = text.lower()

    seq = tokenizer.texts_to_sequences([text])[0]
    seq = pad_sequences([seq], maxlen=max_len, padding="pre")

    pred = model.predict(seq, verbose=0)[0]

    # Don't predict OOV
    pred[1] = 0

    # Apply temperature
    pred = np.log(pred + 1e-10) / temperature
    pred = np.exp(pred)
    pred = pred / np.sum(pred)

    # Top-k sampling
    top_indices = np.argpartition(pred, -top_k)[-top_k:]
    top_probs = pred[top_indices]
    top_probs = top_probs / np.sum(top_probs)

    pred_index = np.random.choice(top_indices, p=top_probs)

    return index_to_word.get(pred_index, "")

In [138]:

seed_text = "what are you"
next_word = predictor(model,tokenize,seed_text,max_len,0.8,5)
print(next_word)

of


In [170]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenize,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [171]:


seed = "artificial intelligence is the subject of the"
generate_text = generate_text(model,tokenize,seed,max_len,5)
print(generate_text)

artificial intelligence is the subject of the city 's liner notes of


In [141]:
print(any("<OOV>" in line for line in text))

False


In [129]:
print(tokenize.word_index["<OOV>"])

1


In [172]:

import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenize, f)

In [173]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)